## IMPORTACION
Librerias, funciones, conexion

In [2]:
import mysql.connector
import pandas as pd
from datetime import datetime
import calendar
from dateutil.relativedelta import relativedelta
from pathlib import Path


In [ ]:
conn = mysql.connector.connect(
    host = "datamart.mex.internal.lyftbikes.com",
    port =3306,
    database= "mex_datawarehouse_bss4",
    user= "dm_mex_ge",
    password= "m+y#J9JI9F+^4qjSJLu^R",
)

conn.cursor().execute(
    "SET SESSION max_execution_time = 300000"
)

In [ ]:
from utils import (
    ejecutar_y_guardar,
    obtener_trimestre_anterior 
)

## FECHA INICIO Y FIN
Fechas que esta tomando en cuenta la consulta

In [5]:
fecha_inicioT, fecha_finT = obtener_trimestre_anterior()

print(fecha_inicioT, fecha_finT)

2026-01-01 00:00:00 2026-04-01 00:00:00


### Ruta
Agregar ruta y nombre del archivo

In [ ]:
ruta_archivo_trimestral = (
    rf"C:\Users\tsunami.martinez\Downloads\Reportes"
    rf"\Trimestre_{fecha_inicioT[:7]}.csv"
)

print(ruta_archivo_trimestral)

## Correr codigo

In [ ]:
query_trimestral = f"""


SELECT  CONCAT(m.firstName, " ", m.lastName) AS nombre_completo,
        m.bikeMemberAttributes_gender AS genero,
        CONVERT_TZ(FROM_UNIXTIME(m.bikeMemberAttributes_birthday/1000), 'UTC', 'America/Mexico_City') AS fecha_nacimiento,
        m.address AS domicilio,
        m.email AS correo_electronico, 
        m.phoneNumber AS tel_contacto,
        IF(ss.id = 1, 'sitio_web', IF(ss.id = 3, 'renov_auto', IF(ss.id = 6, 'app_movil', 'otra')))
          AS forma_inscripcion,
        m.bikeMemberAttributes_memberNumber AS num_pers_usuaria,
        m.currentTransitCardNumber AS num_tarjeta,
        st.name_localizedValue1 AS tipo_membresia

FROM (SELECT id, subscriptionType_id, subscriptionSourceDim_id
      FROM BikeSubscriptionFact
      WHERE start/1000 <= UNIX_TIMESTAMP('{fecha_finT}') -- actualizar fecha a fin del trimestre
      AND end/1000 > UNIX_TIMESTAMP('{fecha_finT}') -- actualizar fecha a fin del trimestre
      AND subscriptionType_id < 5) AS s -- excluye migradas y test product

LEFT JOIN BikeMemberFact AS m ON s.id = m.currentSubscription_id
LEFT JOIN BikeSubscriptionTypeDim AS st ON s.subscriptionType_id = st.id
LEFT JOIN BikeSubscriptionRequestSourceDim AS ss ON s.subscriptionSourceDim_id = ss.id

WHERE m.bikeMemberAttributes_memberNumber IS NOT NULL
;

"""

In [8]:
df_trimestral = ejecutar_y_guardar(
    query_trimestral,
    conn,
    ruta_archivo_trimestral
)

c:\Users\tsunami.martinez\Documents\Querys_Btk\utils.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


Archivo guadado: C:\Users\tsunami.martinez\Downloads\Reportes\Trimestre_2026-01.csv
